# Corpus audit and article embeddings
Mary Kate Carara — modern pipeline. Run from the project uv environment.

The proposal reports 11 attributes and coverage through April 2020; the supplied subset has 15 columns and ends March 26, 2020. `audit_corpus` drops 5 repeated article bodies (keeping the earliest copy), leaving 7,332 records. The regex-filtered corpus describes selected news coverage, not public opinion.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.git').exists())
ROOT = REPO/'final-project/code/modern-methods'
sys.path.insert(0, str(ROOT))
from src import topic_helpers as h
OUT = ROOT/'Results/modern'


In [ ]:
df, summary = h.audit_corpus()
summary

## Inspect the corpus
Read `final-project/data/corpus_review.csv` and fill relevance, boilerplate, malformed, and notes. These are audit flags, not automatic exclusions.

In [ ]:
display(df.sample(10, random_state=42)[['article_id','title','publication','text']])
display(df[['duplicate_body','word_count_difference']].value_counts().head(10))

## Full-text contextual embeddings
The lecture distinguishes contextual sentence embeddings from counts, static word2vec, and arbitrary raw BERT outputs. This notebook uses the chapter’s GTE-small encoder, with 480-token chunks and token-weighted pooling. Every article still contributes one normalized vector. Pooling can blur multiple themes.

Model downloads require network access. Cache manifests pin the resolved revision; interrupted embedding batches resume at article boundaries.

In [ ]:
pilot = df.sample(200, random_state=42).sort_index()
pilot_embeddings, pilot_cache = h.embed_corpus(pilot, ROOT, pilot=True)
display(pd.read_csv(pilot_cache/'chunk_diagnostics.csv').describe())

In [ ]:
embeddings, cache = h.embed_corpus(df, ROOT)
print(embeddings.shape)
display(pd.read_csv(cache/'chunk_diagnostics.csv').describe())
json.loads((cache/'timings.json').read_text())

## Handoff
Laura should use `final-project/data/corpus.csv` and the ordered IDs in `final-project/data/corpus_manifest.json`. Preserve IDs throughout both pipelines. See `notebooks/README.md` for the common export contract and deadlines.